# Bölüm 4/7 — Hiperparametre Optimizasyonu



## Önceki bölümden devam
Bu hücre, `3_...` bölümünde kaydedilen tüm değişkenleri ve tanımlı fonksiyonları geri yükler.

In [ ]:
!pip install dill --quiet
import dill
dill.load_session('checkpoint_3.pkl')
print('Önceki bölümün oturumu yüklendi.')

## Hiperparametre Optimizasyonu

**Kapsam notu:** Hiperparametre araması altı modelin tamamı için yapılmıştır: Linear SVM, Logistic Regression (`C`, `class_weight`), Multinomial NB ve Complement NB (`alpha`), Random Forest ve XGBoost (`n_estimators`, `max_depth`, sınıf dengesizliği ağırlıkları). Böylece aşağıdaki karşılaştırmalar, ayarlanmış ve ayarlanmamış modelleri karıştırmadan tüm modellerin en iyi haliyle yapılmaktadır.

### 1. Linear SVM GridSearchCV

Linear SVM için `GridSearchCV` ile hiperparametre araması yapılır: `C` (düzenlileştirme gücü) ve `class_weight` (sınıf dengesizliği düzeltmesi) parametreleri, 5 katlı çapraz doğrulama ile taranır.

In [ ]:
param_grid = {
    "C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}
grid_svm = GridSearchCV(
    LinearSVC(random_state=42),
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
grid_svm.fit(X_train_tfidf, y_train)
print("Best params:", grid_svm.best_params_)
print("Best CV Macro F1:", grid_svm.best_score_)

GridSearchCV'nin bulduğu en iyi SVM modeli (`best_estimator_`) test setinde değerlendirilir ve sonuçlar (accuracy, precision, recall, F1, confusion matrix) raporlanır.

In [ ]:
best_svm = grid_svm.best_estimator_

y_pred_best_svm = best_svm.predict(X_test_tfidf)

print("Accuracy :", accuracy_score(y_test, y_pred_best_svm))
print("Precision:", precision_score(
    y_test, y_pred_best_svm, pos_label="yes"
))
print("Recall   :", recall_score(
    y_test, y_pred_best_svm, pos_label="yes"
))
print("F1 Score :", f1_score(
    y_test, y_pred_best_svm, pos_label="yes"
))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_svm))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best_svm,
    labels=["no", "yes"]
)

plt.title("Optimized Linear SVM - Confusion Matrix")
plt.show()

### 2. XGBoost GridSearchCV

XGBoost için `GridSearchCV` ile hiperparametre araması yapılır: `n_estimators`, `max_depth`, `learning_rate` ve sınıf dengesizliğini düzeltmeyi hedefleyen `scale_pos_weight` parametreleri taranır.

**Genişletilmiş arama:** İlk taramada en iyi `n_estimators` (300) ve `learning_rate` (0.3) üst sınırda, `max_depth` (3) ise alt sınırda çıkmıştı — üçü birden sınırda çıkması, bu aralığın XGBoost için yetersiz kaldığına işaret eder. Aralıklar buna göre genişletildi. Not: aday sayısı arttığı için bu hücre öncekinden belirgin şekilde daha uzun sürecektir.

In [ ]:
# XGBoost için hiperparametreler
param_grid_xgb = {
    "n_estimators": [200, 300, 400, 500],       # 300 sinirda cikmisti, yukari genisletildi
    "max_depth": [2, 3, 6, 9],                  # 3 sinirda cikmisti, asagi genisletildi
    "learning_rate": [0.05, 0.1, 0.2, 0.3, 0.4, 0.5],  # 0.3 sinirda cikmisti, yukari genisletildi
    "scale_pos_weight": [1, len(y_train_xgb[y_train_xgb==0]) / len(y_train_xgb[y_train_xgb==1])]
}

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_grid=param_grid_xgb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train_tfidf, y_train_xgb)

print("Best Parameters:", grid_xgb.best_params_)
print("Best CV Macro F1:", grid_xgb.best_score_)

En iyi XGBoost modeli test setinde değerlendirilir ve sonuçlar raporlanır.

In [ ]:
best_xgb = grid_xgb.best_estimator_

y_pred_best_xgb = best_xgb.predict(X_test_tfidf)

print("Accuracy :", accuracy_score(y_test_xgb, y_pred_best_xgb))
print("Precision:", precision_score(y_test_xgb, y_pred_best_xgb))
print("Recall   :", recall_score(y_test_xgb, y_pred_best_xgb))
print("F1 Score :", f1_score(y_test_xgb, y_pred_best_xgb))

print("\nClassification Report:\n")
print(
    classification_report(
        y_test_xgb,
        y_pred_best_xgb,
        target_names=["no", "yes"]
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_test_xgb,
    y_pred_best_xgb,
    display_labels=["no", "yes"]
)

plt.title("Optimized XGBoost - Confusion Matrix")
plt.show()

### Train vs Test: Overfitting Kontrolü

`max_depth` genişletilmiş aramada da sınırda (2) çıktığı için, modelin eğitim verisini ezberleyip ezberlemediğini kontrol etmek gerekiyor. Eğer train ve test skorları birbirine yakınsa, model iyi genelliyor demektir ve `reg_alpha`/`reg_lambda` (L1/L2) eklemek muhtemelen gereksizdir. Aradaki fark büyükse (örn. train'de ~1.0, test'te belirgin daha düşük), bu net bir overfitting işaretidir ve düzenlileştirme eklemek gerçekten fayda sağlayabilir.

In [ ]:
y_pred_train_xgb = best_xgb.predict(X_train_tfidf)

train_f1 = f1_score(y_train_xgb, y_pred_train_xgb)
test_f1 = f1_score(y_test_xgb, y_pred_best_xgb)
train_acc = accuracy_score(y_train_xgb, y_pred_train_xgb)
test_acc = accuracy_score(y_test_xgb, y_pred_best_xgb)

print("=== Optimized XGBoost: Train vs Test ===")
print(f"{'Metrik':<12} {'Train':>8} {'Test':>8} {'Fark':>8}")
print(f"{'Accuracy':<12} {train_acc:>8.4f} {test_acc:>8.4f} {train_acc-test_acc:>8.4f}")
print(f"{'F1 (yes)':<12} {train_f1:>8.4f} {test_f1:>8.4f} {train_f1-test_f1:>8.4f}")

print("\nTrain seti classification report:")
print(classification_report(y_train_xgb, y_pred_train_xgb, target_names=["no", "yes"]))


**Yorum:** Train-test farkı küçükse (örn. accuracy farkı ~0.02-0.03 veya altı), model bu sığ ağaç yapısıyla zaten iyi genelliyor demektir ve L1/L2 düzenlileştirme araması muhtemelen zaman kaybı olur. Fark büyükse (örn. train'de neredeyse mükemmel, test'te belirgin düşük), `reg_alpha`/`reg_lambda` parametrelerini mevcut GridSearchCV'ye eklemek mantıklı bir sonraki adımdır.

### XGBoost için L1/L2 Düzenlileştirme (`reg_alpha` / `reg_lambda`)

Train-test farkı (~2 puan accuracy, ~1.5 puan F1) orta düzeyde bir overfitting işareti gösterdiği için `reg_alpha` (L1/Lasso benzeri) ve `reg_lambda` (L2/Ridge benzeri) parametreleri denenir. Bu ikisi, lineer modellerdeki gibi katsayılara değil, her yaprağın (leaf) tahmin skoruna uygulanır — büyük yaprak skorlarını cezalandırarak modelin aşırı özgüvenli bölünmeler yapmasını sınırlar. `reg_lambda` XGBoost'ta varsayılan olarak zaten 1'dir (kısmi L2 zaten devrede), `reg_alpha` varsayılanı 0'dır (L1 hiç yok).

**Kapsam notu:** `n_estimators`, `max_depth`, `learning_rate`, `scale_pos_weight` önceki genişletilmiş aramada zaten iyi değerlerde bulunmuştu; bunları tekrar aramak (4 parametreyi `reg_alpha`/`reg_lambda` ile birlikte) kombinasyon sayısını binlerce fit'e çıkarıp pratik olarak çalıştırılamaz hale getirirdi. Bu yüzden o dört parametre önceki en iyi değerlerinde sabitlenip arama sadece düzenlileştirmeye odaklandı. Bu, ortak (joint) bir aramadan daha az kapsamlı olsa da, hesaplama bütçesini asıl merak edilen soruya (düzenlileştirme overfitting'i azaltıyor mu?) yönlendiren bilinçli bir tercihtir.

In [ ]:
param_grid_xgb_reg = {
    "n_estimators": [400],        # onceki genisletilmis aramanin en iyisi, sabitlendi
    "max_depth": [2],             # onceki genisletilmis aramanin en iyisi, sabitlendi
    "learning_rate": [0.4],       # onceki genisletilmis aramanin en iyisi, sabitlendi
    "scale_pos_weight": [1],      # onceki genisletilmis aramanin en iyisi, sabitlendi
    "reg_alpha": [0, 0.1, 0.5, 1, 5],
    "reg_lambda": [1, 2, 5, 10]
}

grid_xgb_reg = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_grid=param_grid_xgb_reg,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_xgb_reg.fit(X_train_tfidf, y_train_xgb)

print("Best Parameters:", grid_xgb_reg.best_params_)
print("Best CV Macro F1:", grid_xgb_reg.best_score_)
print("Karsilastirma icin onceki (duzenlilestirmesiz) CV Macro F1:", grid_xgb.best_score_)


Düzenlileştirilmiş en iyi model test setinde değerlendirilir ve hem test performansı hem de train-test farkı, düzenlileştirmesiz sürümle karşılaştırılır.

In [ ]:
best_xgb_reg = grid_xgb_reg.best_estimator_

y_pred_best_xgb_reg = best_xgb_reg.predict(X_test_tfidf)
y_pred_train_xgb_reg = best_xgb_reg.predict(X_train_tfidf)

print("=== Optimized XGBoost + L1/L2: Test ===")
print("Accuracy :", accuracy_score(y_test_xgb, y_pred_best_xgb_reg))
print("Precision:", precision_score(y_test_xgb, y_pred_best_xgb_reg))
print("Recall   :", recall_score(y_test_xgb, y_pred_best_xgb_reg))
print("F1 Score :", f1_score(y_test_xgb, y_pred_best_xgb_reg))
print("\nClassification Report:\n")
print(classification_report(y_test_xgb, y_pred_best_xgb_reg, target_names=["no", "yes"]))

ConfusionMatrixDisplay.from_predictions(
    y_test_xgb, y_pred_best_xgb_reg, display_labels=["no", "yes"]
)
plt.title("Optimized XGBoost + L1/L2 - Confusion Matrix")
plt.show()

train_f1_reg = f1_score(y_train_xgb, y_pred_train_xgb_reg)
test_f1_reg = f1_score(y_test_xgb, y_pred_best_xgb_reg)
train_acc_reg = accuracy_score(y_train_xgb, y_pred_train_xgb_reg)
test_acc_reg = accuracy_score(y_test_xgb, y_pred_best_xgb_reg)

print("\n=== Train-Test farki karsilastirmasi ===")
print(f"{'Metrik':<20} {'Duzenlilestirmesiz fark':>24} {'Duzenlilestirmeli fark':>24}")
print(f"{'Accuracy':<20} {train_acc-test_acc:>24.4f} {train_acc_reg-test_acc_reg:>24.4f}")
print(f"{'F1 (yes)':<20} {train_f1-test_f1:>24.4f} {train_f1_reg-test_f1_reg:>24.4f}")


**Yorum:** Eğer düzenlileştirmeli modelin train-test farkı belirgin şekilde küçüldüyse (overfitting azaldıysa) VE test Macro F1'i düşmediyse, bu model bir sonraki adımda ana karşılaştırma tablosuna (`sonuc_tablosu`) eklenmeye adaydır. Fark küçülmediyse veya test performansı belirgin düştüyse, mevcut düzenlileştirmesiz XGBoost'un (zaten iyi genelleyen, sığ ağaçlı) yapısı tercih edilmeye devam edilmelidir.

### 3. Random Forest

Random Forest için `GridSearchCV` ile hiperparametre araması yapılır: `n_estimators`, `max_depth`, `min_samples_leaf` ve `class_weight` parametreleri taranır.

**Genişletilmiş arama:** İlk taramada en iyi `n_estimators` (300) üst sınırda çıkmıştı; 400'e kadar genişletildiğinde CV skoru pratik olarak değişmedi (doygunluk), bu yüzden `n_estimators` tekrar [200, 300]'e küçültüldü. Bunun yerine hiç denenmemiş olan `max_features` parametresi eklendi — TF-IDF gibi ~11.000 boyutlu ve seyrek bir özellik uzayında, her bölünmede kaç özelliğin değerlendirileceği (varsayılan `"sqrt"` ≈ 105 özellik) sonucu belirgin etkileyebilir. `min_samples_leaf` için bulunan değer (1) zaten bu parametrenin alabileceği en küçük geçerli değer olduğundan (0 geçersizdir), bu bir "sınır sorunu" değildir; aynı şekilde `max_depth` için `None` zaten sınırsız derinliği ifade ettiğinden ek bir üst değere gerek yoktur.

In [ ]:
param_grid_rf = {
    "n_estimators": [200, 300],  # 400 denendi, kazanc olmadigi icin kucultuldu (odak: max_features)
    "max_depth": [None, 30],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"],
    "max_features": ["sqrt", "log2", 0.1, 0.3]  # yeni: her bolunmede kac ozellik degerlendirilecek
}
grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid_rf,
    cv=5,
    scoring="f1_macro",
    n_jobs=1,          # RF zaten n_jobs=-1 ile paralel calisiyor, ikisini birden -1 yapmayin
    verbose=1
)
grid_rf.fit(X_train_tfidf, y_train)
print("Best Parameters:", grid_rf.best_params_)
print("Best CV Macro F1:", grid_rf.best_score_)

En iyi Random Forest modeli test setinde değerlendirilir ve sonuçlar raporlanır.

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test_tfidf)
print("Accuracy :", accuracy_score(y_test, y_pred_best_rf))
print("Precision:", precision_score(y_test, y_pred_best_rf, pos_label="yes"))
print("Recall   :", recall_score(y_test, y_pred_best_rf, pos_label="yes"))
print("F1 Score :", f1_score(y_test, y_pred_best_rf, pos_label="yes"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_rf))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best_rf,
    labels=["no", "yes"]
)
plt.title("Optimized Random Forest - Confusion Matrix")
plt.show()

**Random Forest optimizasyonu tamamlandı:** `n_estimators`, `max_depth`, `min_samples_leaf`, `class_weight` ve `max_features` parametrelerinin tamamı denendi ve genişletildi. `max_features` için `"sqrt"` (varsayılan) tüm alternatiflere (`"log2"`, `0.1`, `0.3`) karşı kazandı; `n_estimators` 400'e genişletildiğinde CV skorunda pratik bir değişiklik olmadı. Bu, Random Forest için hiperparametre aramasının artık **doygunluğa ulaştığını** ve ek arama yapmanın düşük getirili olacağını gösteriyor — bu model için optimizasyon burada kapatılmıştır.

### 4. Logistic Regression GridSearchCV

Logistic Regression için `GridSearchCV` ile hiperparametre araması yapılır: `C` (düzenlileştirme gücü) ve `class_weight` (sınıf dengesizliği düzeltmesi) parametreleri, 5 katlı çapraz doğrulama ile taranır.

**Genişletilmiş arama:** İlk taramada en iyi `C` değeri (10) aralığın üst sınırında çıkmıştı — bu, gerçek optimumun daha yüksek bir yerde olabileceğine işaret eder. Aralık 20 ve 50'ye kadar genişletildi.

In [ ]:
param_grid_lr = {
    "C": [0.01, 0.1, 0.5, 1, 2, 5, 10, 20, 50],  # onceki en iyi (10) sinirda cikmisti, yukari genisletildi
    "class_weight": [None, "balanced"]
}
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_lr,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
grid_lr.fit(X_train_tfidf, y_train)
print("Best params:", grid_lr.best_params_)
print("Best CV Macro F1:", grid_lr.best_score_)


GridSearchCV'nin bulduğu en iyi Logistic Regression modeli (`best_estimator_`) test setinde değerlendirilir ve sonuçlar (accuracy, precision, recall, F1, confusion matrix) raporlanır.

In [ ]:
best_lr = grid_lr.best_estimator_

y_pred_best_lr = best_lr.predict(X_test_tfidf)

print("Accuracy :", accuracy_score(y_test, y_pred_best_lr))
print("Precision:", precision_score(y_test, y_pred_best_lr, pos_label="yes"))
print("Recall   :", recall_score(y_test, y_pred_best_lr, pos_label="yes"))
print("F1 Score :", f1_score(y_test, y_pred_best_lr, pos_label="yes"))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_lr))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best_lr,
    labels=["no", "yes"]
)
plt.title("Optimized Logistic Regression - Confusion Matrix")
plt.show()


### 5. Multinomial NB GridSearchCV

Multinomial Naive Bayes için `GridSearchCV` ile `alpha` (Laplace/Lidstone düzgünleştirme parametresi) taranır; küçük `alpha` değerleri modelin eğitim verisindeki nadir kelimelere daha az güvenmesini, büyük değerler ise daha güçlü düzgünleştirme uygulanmasını sağlar. Not: Multinomial NB'de `class_weight` parametresi yoktur, bu yüzden sınıf dengesizliği bu model için `alpha` dışında doğrudan ayarlanamaz.

In [ ]:
param_grid_mnb = {
    "alpha": [0.01, 0.05, 0.1, 0.5, 1, 2, 5]
}
grid_mnb = GridSearchCV(
    MultinomialNB(),
    param_grid_mnb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
grid_mnb.fit(X_train_tfidf, y_train)
print("Best params:", grid_mnb.best_params_)
print("Best CV Macro F1:", grid_mnb.best_score_)


GridSearchCV'nin bulduğu en iyi Multinomial NB modeli test setinde değerlendirilir.

In [ ]:
best_mnb = grid_mnb.best_estimator_

y_pred_best_mnb = best_mnb.predict(X_test_tfidf)

print("Accuracy :", accuracy_score(y_test, y_pred_best_mnb))
print("Precision:", precision_score(y_test, y_pred_best_mnb, pos_label="yes"))
print("Recall   :", recall_score(y_test, y_pred_best_mnb, pos_label="yes"))
print("F1 Score :", f1_score(y_test, y_pred_best_mnb, pos_label="yes"))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_mnb))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best_mnb,
    labels=["no", "yes"]
)
plt.title("Optimized Multinomial NB - Confusion Matrix")
plt.show()


### 6. Complement NB GridSearchCV

Complement Naive Bayes için de aynı şekilde `alpha` parametresi taranır; bu model dengesiz sınıflar için özel olarak tasarlandığından, optimum düzgünleştirme seviyesi Multinomial NB'den farklı çıkabilir.

In [ ]:
param_grid_cnb = {
    "alpha": [0.01, 0.05, 0.1, 0.5, 1, 2, 5]
}
grid_cnb = GridSearchCV(
    ComplementNB(),
    param_grid_cnb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
grid_cnb.fit(X_train_tfidf, y_train)
print("Best params:", grid_cnb.best_params_)
print("Best CV Macro F1:", grid_cnb.best_score_)


GridSearchCV'nin bulduğu en iyi Complement NB modeli test setinde değerlendirilir.

In [ ]:
best_cnb = grid_cnb.best_estimator_

y_pred_best_cnb = best_cnb.predict(X_test_tfidf)

print("Accuracy :", accuracy_score(y_test, y_pred_best_cnb))
print("Precision:", precision_score(y_test, y_pred_best_cnb, pos_label="yes"))
print("Recall   :", recall_score(y_test, y_pred_best_cnb, pos_label="yes"))
print("F1 Score :", f1_score(y_test, y_pred_best_cnb, pos_label="yes"))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_cnb))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best_cnb,
    labels=["no", "yes"]
)
plt.title("Optimized Complement NB - Confusion Matrix")
plt.show()


### 7. NB modelleri için sample_weight (class_weight alternatifi)

Multinomial NB ve Complement NB'de `class_weight` parametresi yoktur, ancak `.fit()` `sample_weight` argümanını kabul eder. Bu, dengesiz sınıflar için manuel bir ağırlıklandırma imkânı sağlar: azınlık sınıfa ("no") daha yüksek ağırlık vererek modelin bu sınıfı ihmal etmesini azaltmayı hedefler. `compute_sample_weight("balanced", y_train)` ile hesaplanan ağırlıklar, aynı `alpha` arama uzayıyla GridSearchCV'ye `fit_params` olarak verilir.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight("balanced", y_train)

grid_mnb_sw = GridSearchCV(
    MultinomialNB(),
    param_grid_mnb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
grid_mnb_sw.fit(X_train_tfidf, y_train, sample_weight=sample_weights)
print("Multinomial NB (sample_weight) -- Best params:", grid_mnb_sw.best_params_)
print("Multinomial NB (sample_weight) -- Best CV Macro F1:", grid_mnb_sw.best_score_)

grid_cnb_sw = GridSearchCV(
    ComplementNB(),
    param_grid_cnb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
grid_cnb_sw.fit(X_train_tfidf, y_train, sample_weight=sample_weights)
print("\nComplement NB (sample_weight) -- Best params:", grid_cnb_sw.best_params_)
print("Complement NB (sample_weight) -- Best CV Macro F1:", grid_cnb_sw.best_score_)


Sample_weight ile eğitilmiş en iyi NB modelleri test setinde değerlendirilir ve alpha-only (sample_weight'siz) sürümleriyle doğrudan karşılaştırılır.

In [ ]:
best_mnb_sw = grid_mnb_sw.best_estimator_
best_cnb_sw = grid_cnb_sw.best_estimator_

y_pred_best_mnb_sw = best_mnb_sw.predict(X_test_tfidf)
y_pred_best_cnb_sw = best_cnb_sw.predict(X_test_tfidf)

print("=== Multinomial NB (alpha + sample_weight) ===")
print("Accuracy :", accuracy_score(y_test, y_pred_best_mnb_sw))
print("Precision:", precision_score(y_test, y_pred_best_mnb_sw, pos_label="yes"))
print("Recall   :", recall_score(y_test, y_pred_best_mnb_sw, pos_label="yes"))
print("F1 Score :", f1_score(y_test, y_pred_best_mnb_sw, pos_label="yes"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_mnb_sw))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best_mnb_sw, labels=["no", "yes"]
)
plt.title("Optimized Multinomial NB + sample_weight - Confusion Matrix")
plt.show()

print("\n\n=== Complement NB (alpha + sample_weight) ===")
print("Accuracy :", accuracy_score(y_test, y_pred_best_cnb_sw))
print("Precision:", precision_score(y_test, y_pred_best_cnb_sw, pos_label="yes"))
print("Recall   :", recall_score(y_test, y_pred_best_cnb_sw, pos_label="yes"))
print("F1 Score :", f1_score(y_test, y_pred_best_cnb_sw, pos_label="yes"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_cnb_sw))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best_cnb_sw, labels=["no", "yes"]
)
plt.title("Optimized Complement NB + sample_weight - Confusion Matrix")
plt.show()


Optimizasyon öncesi ve sonrası accuracy ile "no" sınıfı recall değerleri, üç optimize edilen model (SVM, Random Forest, XGBoost) için karşılaştırmalı çubuk grafiklerle gösterilir.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Optimizasyon oncesi/sonrasi tahminleri karsilastir
karsilastirma = {
    "Linear SVM":    {"once": (y_test, y_pred_svm), "sonra": (y_test, y_pred_best_svm)},
    "Random Forest": {"once": (y_test, y_pred_rf),  "sonra": (y_test, y_pred_best_rf)},
    "XGBoost":       {"once": (y_test_xgb, y_pred_xgb), "sonra": (y_test_xgb, y_pred_best_xgb)},
}

modeller = list(karsilastirma.keys())
once_acc, sonra_acc, once_recall, sonra_recall = [], [], [], []

for isim in modeller:
    for asama, hedef_liste_acc, hedef_liste_recall in [
        ("once", once_acc, once_recall), ("sonra", sonra_acc, sonra_recall)
    ]:
        yt, yp = karsilastirma[isim][asama]
        target_names = ["no", "yes"] if set(np.unique(yt)) == {0, 1} else None
        rapor = classification_report(yt, yp, target_names=target_names, output_dict=True)
        hedef_liste_acc.append(accuracy_score(yt, yp))
        hedef_liste_recall.append(rapor["no"]["recall"])

x = np.arange(len(modeller))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Grafik 1: Accuracy once/sonra ---
ax1 = axes[0]
b1 = ax1.bar(x - width/2, once_acc, width, label="Optimizasyon öncesi", color="#8c9db5")
b2 = ax1.bar(x + width/2, sonra_acc, width, label="Optimizasyon sonrası", color="#2e6f95")
ax1.set_xticks(x); ax1.set_xticklabels(modeller)
ax1.set_ylim(0.90, 1.0)
ax1.set_ylabel("Accuracy")
ax1.set_title("Optimizasyon Öncesi vs Sonrası - Accuracy")
ax1.legend()
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x()+bar.get_width()/2, h+0.002, f"{h:.3f}", ha="center", fontsize=8)

# --- Grafik 2: "no" recall once/sonra ---
ax2 = axes[1]
b3 = ax2.bar(x - width/2, once_recall, width, label="Optimizasyon öncesi", color="#e3a27a")
b4 = ax2.bar(x + width/2, sonra_recall, width, label="Optimizasyon sonrası", color="#c1440e")
ax2.set_xticks(x); ax2.set_xticklabels(modeller)
ax2.set_ylim(0.85, 1.0)
ax2.set_title("Optimizasyon Öncesi vs Sonrası - 'no' Recall")
ax2.legend()
for bars in (b3, b4):
    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x()+bar.get_width()/2, h+0.002, f"{h:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

Optimize edilmiş sekiz model varyantının (SVM, LogReg, RF, XGBoost, iki NB modelinin alpha-only ve alpha+sample_weight sürümleri) accuracy, "no" recall/precision değerlerini bir araya getiren özet bir karşılaştırma tablosu (`sonuc_tablosu`) oluşturulur. Tabloya ayrıca **Macro F1** eklenmiştir: sınıf dağılımı dengesiz olduğu için (yes ≈ çoğunluk sınıfı), yalnızca accuracy'e bakmak yanıltıcı olabilir — Macro F1, her iki sınıfın F1 skorunun ağırlıksız ortalamasını alarak azınlık sınıftaki ("no") performansı da eşit ağırlıkla hesaba katar. Değerler önceki hücrelerdeki gerçek `y_pred_*` değişkenlerinden hesaplanır (sabit/elle girilmiş sayı değildir), böylece notebook yeniden çalıştırıldığında otomatik güncellenir.

In [ ]:
def model_metrikleri(model_adi, y_true, y_pred, pos_label="yes", not_metni=""):
    target_names = ["no", "yes"] if set(pd.unique(y_true)) == {0, 1} else None
    rapor = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
    no_key = "no"
    return {
        "Model": model_adi,
        "Accuracy": accuracy_score(y_true, y_pred),
        "'no' Recall": rapor[no_key]["recall"],
        "'no' Precision": rapor[no_key]["precision"],
        "Macro F1": rapor["macro avg"]["f1-score"],
        "Not": not_metni,
    }

sonuc_tablosu = pd.DataFrame([
    model_metrikleri("Optimized XGBoost", y_test_xgb, y_pred_best_xgb, pos_label=1,
                      not_metni="En yüksek recall, ama en düşük 'no' precision"),
    model_metrikleri("Optimized Random Forest", y_test, y_pred_best_rf,
                      not_metni="Accuracy'de XGBoost'a yakın, daha dengeli"),
    model_metrikleri("Optimized Linear SVM", y_test, y_pred_best_svm,
                      not_metni="En basit/yorumlanabilir, hâlâ güçlü"),
    model_metrikleri("Optimized Logistic Regression", y_test, y_pred_best_lr,
                      not_metni="Optimizasyonla dengesizlik duzeltmesi (class_weight) denendi"),
    model_metrikleri("Optimized Complement NB", y_test, y_pred_best_cnb,
                      not_metni="Dengesiz veri için tasarlanmış, alpha optimize edildi"),
    model_metrikleri("Optimized Multinomial NB", y_test, y_pred_best_mnb,
                      not_metni="Alpha optimize edildi, ama sinif agirligi ayarlanamiyor"),
    model_metrikleri("Complement NB + sample_weight", y_test, y_pred_best_cnb_sw,
                      not_metni="class_weight yerine sample_weight ile dengesizlik duzeltmesi"),
    model_metrikleri("Multinomial NB + sample_weight", y_test, y_pred_best_mnb_sw,
                      not_metni="class_weight yerine sample_weight ile dengesizlik duzeltmesi"),
])

# Siralama artik Macro F1'e gore -- dengesiz siniflarda accuracy'den daha guvenilir
sonuc_tablosu = sonuc_tablosu.sort_values("Macro F1", ascending=False).reset_index(drop=True)
sonuc_tablosu.round(4)


### ROC ve Precision-Recall Eğrileri

Yukarıdaki tabloya göre **Macro F1'i en yüksek olan optimize model** otomatik olarak seçilir ve bu model için eşik-bağımsız iki performans eğrisi çizilir:

- **ROC Eğrisi (AUC):** modelin farklı karar eşiklerinde sınıfları ne kadar iyi ayırt ettiğini tek bir sayıya indirger.
- **Precision-Recall Eğrisi:** sınıflar dengesiz olduğu için (`yes` çoğunlukta), çoğu zaman ROC'tan daha bilgilendirici kabul edilir; azınlık sınıfa (`no`) odaklanmak için `pos_label="no"` ile çizilmiştir.

In [ ]:
# En iyi modeli (Macro F1'e gore) sonuc_tablosu'ndan otomatik sec
en_iyi_model_adi = sonuc_tablosu.iloc[0]["Model"]
print("ROC/PR icin secilen model:", en_iyi_model_adi, "| Macro F1:", round(sonuc_tablosu.iloc[0]["Macro F1"], 4))

model_obj_map = {
    "Optimized XGBoost": (best_xgb, y_test_xgb),
    "Optimized Random Forest": (best_rf, y_test),
    "Optimized Linear SVM": (best_svm, y_test),
    "Optimized Logistic Regression": (best_lr, y_test),
    "Optimized Complement NB": (best_cnb, y_test),
    "Optimized Multinomial NB": (best_mnb, y_test),
    "Complement NB + sample_weight": (best_cnb_sw, y_test),
    "Multinomial NB + sample_weight": (best_mnb_sw, y_test),
}

en_iyi_model, en_iyi_y_true = model_obj_map[en_iyi_model_adi]
# ROC/PR icin "no" sinifi pozitif etiket olarak alinir (azinlik sinifi, daha bilgilendirici)
# XGBoost 0/1 etiketli oldugu icin "no" siniftaki gibi davranmak istiyorsak once ceviri gerekir
if en_iyi_model_adi == "Optimized XGBoost":
    # XGBoost'ta 0 = "no", 1 = "yes" -- "no" sinifini pozitif almak icin etiketleri ters cevirelim
    proba = en_iyi_model.predict_proba(X_test_tfidf)[:, 0]  # "no" (0) sinifi olasiligi
    y_true_no = (en_iyi_y_true == 0).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fpr, tpr, _ = roc_curve(y_true_no, proba)
    axes[0].plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.3f}", color="#c0392b")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title(f"ROC Eğrisi — {en_iyi_model_adi} ('no' pozitif)")
    axes[0].legend()

    prec, rec, _ = precision_recall_curve(y_true_no, proba)
    ap_skoru = average_precision_score(y_true_no, proba)
    axes[1].plot(rec, prec, color="#2e6f95", label=f"AP = {ap_skoru:.3f}")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title(f"Precision-Recall Eğrisi — {en_iyi_model_adi} ('no' pozitif)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    RocCurveDisplay.from_estimator(en_iyi_model, X_test_tfidf, en_iyi_y_true, pos_label="no", ax=axes[0])
    axes[0].set_title(f"ROC Eğrisi — {en_iyi_model_adi} ('no' pozitif)")

    PrecisionRecallDisplay.from_estimator(en_iyi_model, X_test_tfidf, en_iyi_y_true, pos_label="no", ax=axes[1])
    axes[1].set_title(f"Precision-Recall Eğrisi — {en_iyi_model_adi} ('no' pozitif)")
    plt.tight_layout()
    plt.show()


## Bu bölümü kaydet
Bir sonraki bölümün bu noktadan devam edebilmesi için tüm oturum (değişkenler, modeller, fonksiyonlar) diske kaydedilir.

In [ ]:
import dill
dill.dump_session('checkpoint_4.pkl')
print('Oturum checkpoint_4.pkl olarak kaydedildi.')